In [1]:
# =============================================
# Generate Current Data and Forecasts
# =============================================



# ---------------------------------------------
# 1. Load Libraries & Config Files
# ---------------------------------------------

# File and API handling
import requests
from pathlib import Path

# Data handling
import pandas as pd 
import numpy as np 
import math
import statistics
from datetime import datetime, timezone
import matplotlib.pyplot as plt



# ---------------------------------------------
# 2. Configuration
# ---------------------------------------------

LOCATION = "London"
LAT = 51.505 
LON = 0.05500000000000002 

SENSOR_URL = "https://aviationweather.gov/api/data/metar"
SENSOR_STATION = "EGLC"
SENSOR_CSV = f"{SENSOR_STATION}_metar_current.csv"
SENSOR_params = {
    "ids": SENSOR_STATION,
    "format": "json",
    "hours": 24
}

FORECAST_URL = "https://api.open-meteo.com/v1/forecast" 



# ---------------------------------------------
# 3. Fetch Live Sensor Data
# ---------------------------------------------
resp = requests.get(SENSOR_URL, params=SENSOR_params)
resp.raise_for_status() 

df_sensor = pd.DataFrame(resp.json()) 

if len(df_sensor) == 0:
    print("No data returned") 
    exit() 

# Time normalization
df_sensor["obsTime"] = pd.to_datetime(df_sensor["obsTime"], unit="s", utc=True)

# Rename to canonical schema
df_sensor = df_sensor.rename(columns={
    "temp": "temp_c",
    "dewp": "dewpoint_c",
    "wdir": "wind_dir",
    "wspd": "wind_speed",
    "wgst": "wind_gust",
    "altim": "pressure"
})

# Add source tag
df_sensor["source"] = "aviationweather"

# Ensure canonical columns exist
canonical_cols = [
    "obsTime",
    "temp_c",
    "dewpoint_c",
    "wind_dir",
    "wind_speed",
    "wind_gust",
    "pressure",
    "source"
]
for col in canonical_cols:
    if col not in df_sensor.columns:
        df_sensor[col] = pd.NA
df_sensor = df_sensor[canonical_cols]


# ---------------------------------------------
# 4. Write to CSV file
# ---------------------------------------------

# Deduplicate
df_sensor = df_sensor.drop_duplicates(subset=["obsTime"], keep="last")

# Sort chronologically 
df_sensor = df_sensor.sort_values("obsTime") 

# ISO format for stability 
df_sensor["obsTime"] = df_sensor["obsTime"].dt.strftime("%Y-%m-%dT%H:%M:%SZ") 

# Write CSV
df_sensor.to_csv(SENSOR_CSV, index=False)



# ---------------------------------------------
# 5. OUTPUT SUMMARY
# ---------------------------------------------
print("Sensor Data Updated")

Sensor Data Updated
